# Reproducing Seiler et al. (2022) from dandiset 000971

Paper. Seiler JL, Cosme CV, Sherathiya VN, Schaid MD, Bianco JM, Bridgemohan AS, Lerner TN (2022).
Dopamine signaling in the dorsomedial striatum promotes compulsive behavior. *Current Biology* 32,
1175-1188. [doi:10.1016/j.cub.2022.01.055](https://doi.org/10.1016/j.cub.2022.01.055). This attempt
used the full text of the accepted author manuscript in PubMed Central (PMC8930615). The supplementary
figures could not be downloaded, so claims on Figures S1 to S7 rest on the main text.

Dandiset. [DANDI:000971](https://dandiarchive.org/dandiset/000971/0.261013.1707), version
0.261013.1707. The paper predates the dandiset and cites no version. This is the most recent published
version. All three published versions (0.240802.2004, 0.260213.1851, 0.261013.1707) and the draft hold
the same 4139 assets with the same blobs; they differ only in dandiset metadata (contributors, ORCIDs,
study targets), none of which the analysis uses.

Code read. The authors' photometry code (GitHub `jillianseiler/Seiler-et-al.-2022-MATLAB-code`,
commit 56f1529) and the conversion code with its notes (GitHub `catalystneuro/lerner-lab-to-nwb`,
commit d301f7e). The photometry processing here follows the authors' code where the Methods are silent.

Outside inputs. One claim (Figure S3A-B) uses the densitometry table on Zenodo
(doi:10.5281/zenodo.11266654), which the dandiset names as its supplement.

| Claim | Tested |
|---|---|
| fig1d-ri60-vs-rr20-shocks | tested |
| fig1e-omission-ri60-longer | tested |
| fig1f-omission-shock-no-correlation | tested |
| fig2a-phenotype-rule | tested |
| fig2b-only-dpr-increase | tested |
| fig2c-nosepoke-rate | tested |
| fig2d-omission-pr-vs-ps | tested |
| fig2k-ps-wait-longer | tested |
| fig2l-extra-nosepokes | tested |
| fig2m-inter-reward-interval | tested |
| fig2n-rewards-per-session | tested |
| fig2o-escalation-timing | tested |
| figs1i-ri60-more-rewards | tested |
| figs2b-males-more-shocks | tested |
| fig3e-ri30-dms-peaks | blocked: no RI30 photometry in the dandiset |
| fig3f-dms-score-phenotype | tested |
| fig3g-dms-score-shock-correlation | tested |
| fig3h-dms-score-omission-no-correlation | tested |
| fig4f-dls-score-no-phenotype | tested |
| fig4g-dls-score-shock-no-correlation | tested |
| fig4h-dls-score-omission-no-correlation | tested |
| fig5f-dms-chr2-escalation | tested |
| fig5g-dms-chr2-more-shocks-early | tested |
| fig5h-dms-chr2-mostly-pr | tested |
| figs5f-chr2-pr-by-sex | tested |
| fig5i-dms-stim-no-omission-effect | tested |
| fig6-nphr-learning-dropout | tested |
| fig6g-nphr-fewer-shocks-late | tested |
| fig6h-nphr-more-ps | tested |
| fig6i-nphr-no-omission-effect | tested |
| fig7f-dls-escalation-similar | tested |
| fig7g-dls-stim-no-shock-effect | tested |
| fig7h-dls-similar-phenotypes | tested |
| fig7i-dls-no-omission-effect | tested |
| figs3ab-dat-protein-not-reduced | tested on Zenodo data |
| figs1a-shock-aversive | blocked: fear conditioning not in the dandiset |
| figs3cd-open-field | blocked: open field not in the dandiset |
| fig3c-gcamp-in-dopamine-neurons | blocked: histology not in the dandiset |

In [ ]:
import hashlib
import io
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import seiler

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

started = time.time()
sessions = seiler.list_sessions()
print(f"{len(sessions)} assets in dandiset {seiler.DANDISET_ID} version {seiler.DANDISET_VERSION}")

# Each subject's experiment and arm, from its files other than the 'Unknown' DLS files.
named = sessions[(sessions.arm != "Unknown") & sessions.experiment.notna()]
subj_arm = named.groupby("subject").group.agg(lambda g: g.value_counts().index[0])
sessions["subject_group"] = sessions.subject.map(subj_arm)
sessions["subject_experiment"] = sessions.subject_group.str.extract(
    r"^(FP|Opto-DLS-Excitatory|Opto-DMS-Excitatory|Opto-DMS-Inhibitory)")[0]
sessions["subject_arm"] = sessions.subject_group.str.split("-").str[-1]

is_fp = sessions.subject_experiment == "FP"
is_opto = sessions.subject_experiment.str.startswith("Opto", na=False)
need = is_fp | (is_opto & sessions.stage.isin(["shock", "omission"])) | (
    sessions.subject_experiment.isin(["Opto-DMS-Excitatory", "Opto-DLS-Excitatory"]) & sessions.stage.isin(["FR1", "RI30"]))
# one file of every optogenetic subject, for its notes
need |= is_opto & ~sessions.subject.duplicated() & sessions.subject_experiment.notna()
to_load = sessions[need]
t = time.time()
events = seiler.read_all_events(to_load.asset_id.tolist())
print(f"read the events of {len(events)} files in {time.time() - t:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
subjects = sessions[sessions.subject_experiment.notna()].drop_duplicates("subject")
print("Subjects by experiment and arm (arm from the subject's files other than the 'Unknown' DLS files):")
print(pd.crosstab(subjects.subject_group, subjects.sex).to_string())
print("\nWestern blot assets:", int((sessions.group == "western_blot").sum()))
print("Files by experiment and stage:")
print(pd.crosstab(sessions.subject_experiment.fillna("western_blot"), sessions.stage).to_string())
two = sessions[sessions.experiment.notna() & (sessions.arm != "Unknown")].groupby("subject").experiment.nunique()
print("\nSubjects with files in two experiments:", two[two > 1].index.tolist())
for s in two[two > 1].index:
    print(" ", s, sessions[sessions.subject == s].group.value_counts().to_dict())
print("Paper: fiber photometry 36 RI60 mice (9 PR, 9 DPR, 18 PS) and RR20 9 (Fig 1D) or 7 (Methods);"
      " DMS excitatory 39 mice, DMS inhibitory 26 (Methods) or 47 (Fig 6F), DLS excitatory 36.")

The western blot assets hold one image per condition and no measurements. The fiber photometry arm of
each mouse is in its file names; a subject named in two experiments is assigned to the experiment of
most of its files.

In [ ]:
def event_hash(ev):
    h = hashlib.md5()
    for k in sorted(ev):
        h.update(k.encode())
        h.update(np.round(np.asarray(ev[k]), 3).tobytes())
    return h.hexdigest()

loaded = sessions[sessions.asset_id.isin(list(events))].copy()
loaded["hash"] = [event_hash(events[a]["events"]) if events[a]["events"] else "" for a in loaded.asset_id]
dups = loaded[(loaded.hash != "") & loaded.duplicated("hash", keep=False)].sort_values("hash")
pairs = dups.groupby("hash").agg(n=("path", "size"), subjects=("subject", "nunique"),
                                 groups=("group", lambda g: ",".join(sorted(set(g)))), stage=("stage", "first"),
                                 n_events=("asset_id", lambda a: sum(len(v) for v in events[a.iloc[0]]["events"].values())))
print("Loaded files whose event series are identical to another loaded file:", len(dups), "in", len(pairs), "sets")
print(pairs.groupby(["subjects", "groups", "stage"]).size().to_string())
exp_files = sessions[sessions.experiment.notna()]
same_start = exp_files[exp_files.duplicated(["subject", "start"], keep=False)]
print("\nFiles in the 'Unknown' DLS excitatory group:", int((sessions.arm == "Unknown").sum()))
print("All files sharing subject and start time with another file:", len(same_start),
      "; their groups:", same_start.group.value_counts().to_dict())
odd = pairs[pairs.subjects > 1]
for h in odd.index:
    d = dups[dups.hash == h]
    r = seiler.rewards(events[d.asset_id.iloc[0]]["events"])
    print("\nIdentical across subjects:", d.subject.tolist(), d.stage.iloc[0], "; reward intervals (s) from",
          np.round(np.diff(r).min(), 2), "to", np.round(np.diff(r).max(), 2), "; series:", list(events[d.asset_id.iloc[0]]["events"]))

The 'Unknown' DLS excitatory files repeat sessions of the same mice that are also deposited under their
named arm, with the same start time and identical events; each such session is counted once below
(the named copy is kept). The omission probes identical across three mice hold only reward times, at
the 20 s spacing the omission schedule gives a mouse that never nosepokes, so they are consistent
with three mice that did not respond, not with one file deposited three times.

In [ ]:
ok_loaded = loaded[~((loaded.arm == "Unknown") & loaded.duplicated(["subject", "start"], keep=False))].copy()
fp = sessions[sessions.subject_experiment == "FP"]
fp_subj = fp.drop_duplicates("subject").copy()
fp_subj["label"] = [seiler.note_field(events[a]["notes"], "Punishment Group") for a in fp_subj.asset_id]
fp_subj["behavior"] = [seiler.note_field(events[a]["notes"], "Behavior") for a in fp_subj.asset_id]
fp_subj["did_not_learn"] = [seiler.note_field(events[a]["notes"], "Did Not Learn") for a in fp_subj.asset_id]
PHENO = {"Punishment Resistant": "PR", "Delayed Punishment Resistant": "DPR", "Punishment Sensitive": "PS"}
fp_subj["pheno"] = np.where(fp_subj.behavior == "RI60", fp_subj.label.map(PHENO), "RR20")
print("Fiber photometry mice: file-name arm, stored punishment group and schedule")
print(pd.crosstab([fp_subj.subject_arm, fp_subj.behavior], fp_subj.label).to_string())
print("Labels consistent across each mouse's files:",
      all(len({seiler.note_field(events[a]["notes"], "Punishment Group") for a in fp[fp.subject == s].asset_id}) == 1
          for s in fp_subj.subject))
opto_subj = sessions[sessions.subject_experiment.str.startswith("Opto", na=False)].drop_duplicates("subject").copy()
opto_subj["label"] = [seiler.note_field(events[a]["notes"], "Punishment Group") for a in opto_subj.asset_id]
opto_subj["did_not_learn"] = [seiler.note_field(events[a]["notes"], "Did Not Learn") for a in opto_subj.asset_id]
print("\nOptogenetic mice, stored punishment group:", opto_subj.label.value_counts().to_dict())
print("Did Not Learn flags, all mice:", pd.concat([fp_subj, opto_subj]).groupby("subject_group").did_not_learn.value_counts().unstack(fill_value=0).to_string())

The fiber photometry mice carry their phenotype in two places, the file names and the notes, and the two
agree. RR20 mice are labelled 'Punishment Sensitive' in the notes; the paper does not classify RR20
mice. Optogenetic mice have no stored phenotype, so they are classified here by the Methods rule.

In [ ]:
side_rows = []
for a, r in ok_loaded.set_index("asset_id").iterrows():
    if not isinstance(r.side, str) or r.stage not in ("FR1", "RI30", "RI60", "RR20", "shock"):
        continue
    ev = events[a]["events"]
    rw = seiler.rewards(ev)
    if len(rw) == 0:
        continue
    def frac(s):
        p = ev.get(f"{s}_nose_poke_times", np.array([]))
        return np.isin(np.round(rw, 2), np.round(p, 2)).mean() if len(p) else 0.0
    other = "left" if r.side == "right" else "right"
    series = "+".join(k.split("_")[0] for k in ("left_reward_times", "right_reward_times") if k in ev)
    side_rows.append({"stage": r.stage, "described_side": r.side, "reward_series": series,
                      "rewards_in_described_side_pokes": frac(r.side) > 0.9, "rewards_in_other_side_pokes": frac(other) > 0.9})
side_df = pd.DataFrame(side_rows)
print("Do reward times coincide with nosepokes on the side the session description names? (files with rewards)")
print(side_df.groupby(["stage", "described_side", "reward_series", "rewards_in_described_side_pokes",
                       "rewards_in_other_side_pokes"]).size().to_string())

In training and probe sessions the reward times coincide with nosepokes on the side the description
names. In shock probes on the left side the reward series is stored as `right_reward_times` while its
times are left nosepokes, so the series name is wrong there; rewards are taken from either series. In
most RR20 files the reward times do not coincide with any nosepoke time (MedPC delivers the RR reward
after the counted nosepoke); RR20 files are used here only for reward counts and probes.

In [ ]:
probe_files = ok_loaded[ok_loaded.stage == "shock"]
chk = []
for a in probe_files.asset_id:
    ev = events[a]["events"]
    if "footshock_times" not in ev:
        continue
    sh, rw = ev["footshock_times"], seiler.rewards(ev)
    chk.append({"n_shock": len(sh), "n_rew": len(rw), "shock_at_reward": np.isin(np.round(sh, 2), np.round(rw, 2)).mean() if len(sh) else np.nan,
                "first5_unshocked": not np.isin(np.round(rw[:5], 2), np.round(sh, 2)).any(), "last_event_s": max(v.max() for v in ev.values() if len(v))})
chk = pd.DataFrame(chk)
print(f"Shock probe files with footshock_times: {len(chk)} of {len(probe_files)}")
print(f"Fraction of shocks at a reward time: median {chk.shock_at_reward.median():.2f}, min {chk.shock_at_reward.min():.2f}")
ratio = chk.n_shock / chk.n_rew.replace(0, np.nan)
print(f"Shocks per reward: median {ratio.median():.2f} (Methods: RR3, about 1/3); 5th to 95th percentile {ratio.quantile(.05):.2f} to {ratio.quantile(.95):.2f}")
print("First five rewards never shocked (Methods):", int(chk.first5_unshocked.sum()), "of", len(chk))
print(f"Last event time in probe files: up to {chk.last_event_s.max():.0f} s (Methods: 60 min or 10 min of inactivity)")
dur = []
for a, st in ok_loaded.set_index("asset_id").stage.items():
    ev = events[a]["events"]
    if ev and st in ("FR1", "RI30", "RI60", "RR20", "omission"):
        dur.append({"stage": st, "last_event_s": max((v.max() for v in ev.values() if len(v)), default=np.nan), "n_rewards": len(seiler.rewards(ev))})
dur = pd.DataFrame(dur)
print("\nLast event time (s) and reward count by stage:")
print(dur.groupby("stage").agg(files=("n_rewards", "size"), last_event_max=("last_event_s", "max"), last_event_median=("last_event_s", "median"),
                               rewards_max=("n_rewards", "max")).round(0).to_string())

Every shock falls on a rewarded nosepoke, about one reward in three is shocked, and the first five
rewards are never shocked, as the Methods describe. Sessions hold at most 50 rewards and end within
about an hour, consistent with the Methods' one hour or 50 rewards; the exception is an RI60 file whose
last event is at 5354 s, which may be one of the sessions the conversion notes say were stitched from two
recordings. Times are in seconds from the session start.

In [ ]:
fp_probes = seiler.probe_table(fp, events)
print("Probe days per fiber photometry mouse:", fp_probes.groupby("subject").size().value_counts().sort_index().to_dict())
print("Probe days with more than one file:", int((fp_probes.n_files > 1).sum()), "; with more than one file holding footshock_times:",
      int((fp_probes.n_with_shocks > 1).sum()))
print(fp_probes[fp_probes.n_with_shocks > 1].to_string(index=False))
print("\nTraining sessions before the early and late probes (Results: 1 to 2 and 13 to 14 days):")
print(fp_probes[fp_probes.probe != ""].groupby("probe").train_days_before.describe().to_string())
fp_el = seiler.early_late(fp_probes).join(fp_subj.set_index("subject")[["pheno", "label", "behavior", "sex"]])
print("\nMice missing a probe:")
print(fp_el[fp_el[["early", "late"]].isna().any(axis=1)].to_string())
print("Shock counts, range:", fp_el[["early", "late"]].min().min(), "to", fp_el[["early", "late"]].max().max())

Shock probes come in the order the Results give, the early one after 0 to 4 RI60 (or RR20) sessions
(median 1) and the late one after 11 to 18 (median 13). Two probe days have two files that both hold
shocks; the last of the day is used, and the sum is a variant. Five mice lack a probe in the dandiset:
two PR mice (028.392, 62.207) and one RR20 mouse have no late probe, and one PR mouse (89.247) and one
PS mouse (78.214) have no early probe. For 89.247 a photometry session one day after its first RI60
session is deposited as a 'Probe Test' session with no footshock series. 32 RI60 mice and 5 RR20 mice
have both probes.

In [ ]:
all_ph = fp[fp.asset_id.map(lambda a: events[a]["has_photometry"])]
print("Files with photometry by stage:", all_ph.stage.value_counts().to_dict())
fp_sessions = all_ph[all_ph.stage.isin(["RI60", "RR20"])].sort_values("start_ts").copy()
fp_first_probe = fp_probes.sort_values("date").groupby("subject").date.first()
ri60_dates = fp[fp.stage.isin(["RI60", "RR20"])].groupby("subject").date.apply(sorted)
fp_sessions["train_day"] = [sum(d < r.date for d in ri60_dates.get(r.subject, [])) + 1 for r in fp_sessions.itertuples()]
early_probe = fp_probes[fp_probes.probe == "early"].set_index("subject").date
fp_sessions["fp_stage"] = [
    ("early" if r.date < early_probe[r.subject] else "late") if r.subject in early_probe.index
    else ("early" if r.train_day <= 3 else "late") for r in fp_sessions.itertuples()]
fp_sessions = fp_sessions.merge(fp_subj[["subject", "pheno", "behavior"]], on="subject")
print("Photometry sessions:", len(fp_sessions), "from", fp_sessions.subject.nunique(), "mice")
print(pd.crosstab([fp_sessions.pheno, fp_sessions.stage], fp_sessions.fp_stage).to_string())
print("\nMice with photometry per phenotype and stage (paper Fig 3E: RI30 PR 6, DPR 6, PS 11; early PR 7, DPR 9, PS 15; late PR 5, DPR 6, PS 13):")
print(fp_sessions.groupby(["pheno", "fp_stage"]).subject.nunique().unstack().to_string())
print("\nTraining day of each photometry session (1 = first RI60 or RR20 session):")
print(fp_sessions.groupby("fp_stage").train_day.describe().to_string())


def process_photometry(row):
    rec = seiler.load_photometry(row.asset_id)
    d, fs, t0, ev = rec["data"], rec["fs"], rec["t0"], rec["events"]
    rew, unrew = seiler.split_nosepokes(ev, row.side)
    out = {"asset_id": row.asset_id, "fs": fs, "t0": t0, "n_samples": len(d), "duration_s": len(d) / fs,
           "last_event_s": max((v.max() for v in ev.values() if len(v)), default=np.nan),
           "n_rewards": len(seiler.rewards(ev)), "locations": ",".join(rec["locations"]),
           "sources": ",".join(s.split("_")[-1] for s in rec["sources"]),
           "n_rewarded": len(rew), "n_unrewarded": len(unrew), "coordinates": str(np.round(rec["coordinates"], 2).tolist()),
           "modulation": ";".join(f"{k.replace('commanded_voltage_series_', '')}={v:g}" for k, v in sorted(rec["modulation_hz"].items()))}
    for region, (si, ci) in {"DMS": (0, 1), "DLS": (2, 3)}.items():
        out[f"{region}_dropout_frac"] = seiler.dropout_mask(d[:, si], d[:, ci], fs).mean()
        z = seiler.zscored_dff(d[:, si], d[:, ci], fs)
        res = seiler.peak_score(z, fs, rew, unrew, t0)
        out[f"{region}_score"], out[f"{region}_rew_max"], out[f"{region}_unrew_min"] = res["score"], res["rew_max"], res["unrew_min"]
        out[f"{region}_psth_rew"], out[f"{region}_psth_unrew"] = res["psth_rew"], res["psth_unrew"]
        out[f"{region}_auc_2_10"] = res["auc_2_10"]
        # the authors' processing without the dropout step
        z0 = seiler.zscored_dff(d[:, si], d[:, ci], fs, mask_dropouts=False)
        out[f"{region}_score_nomask"] = seiler.peak_score(z0, fs, rew, unrew, t0)["score"]
        out[f"{region}_max_abs_z_nomask"] = np.nanmax(np.abs(z0))
        # clock check: the same events shifted by 37 s
        out[f"{region}_rew_max_shifted"] = seiler.peak_score(z, fs, rew + 37.0, unrew + 37.0, t0)["rew_max"]
        # column check: the channels the other way round
        zs = seiler.zscored_dff(d[:, ci], d[:, si], fs)
        out[f"{region}_rew_max_swapped"] = seiler.peak_score(zs, fs, rew, unrew, t0)["rew_max"]
    out["t_psth"] = res["t"]
    return out


t = time.time()
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(6) as ex:
    ph = pd.DataFrame(list(ex.map(process_photometry, fp_sessions.itertuples())))
ph = fp_sessions.merge(ph, on="asset_id")
# Methods: recordings with excessive artifacts were excluded; here a region's session is excluded when
# more than 10% of its samples are dropouts (a choice here).
for region in ["DMS", "DLS"]:
    bad = ph[f"{region}_dropout_frac"] > 0.10
    ph.loc[bad, [f"{region}_score", f"{region}_rew_max", f"{region}_unrew_min", f"{region}_score_nomask"]] = np.nan
    print(f"{region} sessions excluded for dropouts over 10% of samples:", ph.loc[bad, ["subject", "date"]].to_dict("records"))
print(f"\nprocessed {len(ph)} photometry sessions in {time.time() - t:.0f} s")
print("Sampling rate (Hz):", ph.fs.unique(), "; start time:", ph.t0.unique())
print("Column locations:", list(ph.locations.unique()), "; excitation sources:", list(ph.sources.unique()))
print("Modulation frequencies (Hz) in the files:", ph.modulation.value_counts().to_dict())
print("Fiber coordinates (AP, ML, DV) per column in the files:", ph.coordinates.value_counts().to_dict(),
      "; Methods: DMS AP 0.8, ML 1.5, DV -2.8; DLS AP -0.1, ML 2.8, DV -3.5")
print(f"Recording length minus last behavioral event (s): {np.nanmin(ph.duration_s - ph.last_event_s):.0f} to {np.nanmax(ph.duration_s - ph.last_event_s):.0f}")
print(f"Rewarded / unrewarded nosepokes per session: {ph.n_rewarded.min()} to {ph.n_rewarded.max()} / {ph.n_unrewarded.min()} to {ph.n_unrewarded.max()}")
print("Sessions with rewards but no reward at a nosepoke time:", ph[(ph.n_rewarded == 0)][["subject", "date", "n_rewards"]].to_dict("records"))
for region in ["DMS", "DLS"]:
    m = ph[f"{region}_dropout_frac"] > 0
    fr = ph[f"{region}_dropout_frac"]
    print(f"{region}: percent of samples left out per session, median {100 * fr.median():.3f}, above 0.1% in {(fr > 0.001).sum()} sessions, "
          f"maximum {100 * fr.max():.2f}; largest |z| without the dropout step {ph[f'{region}_max_abs_z_nomask'].max():.0f} "
          f"(above 50 in {(ph[f'{region}_max_abs_z_nomask'] > 50).sum()} sessions)")

In [ ]:
chk = pd.DataFrame({
    "region": np.repeat(["DMS", "DLS"], len(ph)),
    "as_labelled": np.r_[ph.DMS_rew_max, ph.DLS_rew_max],
    "events_shifted_37s": np.r_[ph.DMS_rew_max_shifted, ph.DLS_rew_max_shifted],
    "channels_swapped": np.r_[ph.DMS_rew_max_swapped, ph.DLS_rew_max_swapped]})
print("Maximum of the rewarded nosepoke PSTH in 0 to 1.5 s (z), median [5th, 95th percentile] over sessions:")
print(chk.groupby("region").agg(lambda x: f"{np.nanmedian(x):.2f} [{np.nanpercentile(x, 5):.2f}, {np.nanpercentile(x, 95):.2f}]").to_string())
for region in ["DMS", "DLS"]:
    a, b = ph[f"{region}_rew_max"], ph[f"{region}_rew_max_shifted"]
    ok = a.notna() & b.notna()
    print(f"{region}: as labelled above shifted in {(a[ok] > b[ok]).sum()} of {ok.sum()} sessions, Wilcoxon p = {stats.wilcoxon(a[ok], b[ok]).pvalue:.2g}")
    c = ph[f"{region}_rew_max_swapped"]
    print(f"{region}: as labelled above swapped in {(a[ok] > c[ok]).sum()} of {ok.sum()} sessions")
a_dms, a_dls = ph.DMS_auc_2_10, ph.DLS_auc_2_10
ok = a_dms.notna() & a_dls.notna()
print(f"Rewarded PSTH area 2 to 10 s after the nosepoke (z s), median DMS {a_dms.median():.2f}, DLS {a_dls.median():.2f}; "
      f"DLS above DMS in {(a_dls[ok] > a_dms[ok]).sum()} of {ok.sum()} sessions, Wilcoxon p = {stats.wilcoxon(a_dls[ok], a_dms[ok]).pvalue:.2g}")
print("Sessions whose rewarded maximum is below 0.3 z in both regions:", ph[(ph.DMS_rew_max < .3) & (ph.DLS_rew_max < .3)].subject.tolist())
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
tt = ph.t_psth.iloc[0]
for ax, region in zip(axes, ["DMS", "DLS"]):
    for lab, c in [("PR", "k"), ("DPR", "m"), ("PS", "c"), ("RR20", "y")]:
        sel = ph[ph.pheno == lab]
        ax.plot(tt, np.nanmean(np.stack(sel[f"{region}_psth_rew"]), 0), c, label=f"{lab} rewarded (dashed: unrewarded)")
        ax.plot(tt, np.nanmean(np.stack(sel[f"{region}_psth_unrew"]), 0), c + "--")
    ax.axvline(0, color="grey", lw=.5); ax.set_title(region); ax.set_xlabel("s from nosepoke")
axes[0].set_ylabel("z"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()
allp = np.concatenate([np.stack(ph[f"{r}_psth_{k}"]).ravel() for r in ["DMS", "DLS"] for k in ["rew", "unrew"]])
print(f"Plotted session-mean PSTHs span {np.nanmin(allp):.2f} to {np.nanmax(allp):.2f} z before averaging over mice")

The dandiset holds photometry for RI60 and RR20 training sessions only (and one session described as a
probe test, not used): none for RI30 and none for the shock probes, though the Methods say recordings
were made during RI30 and on both probes. Early sessions are on the first to third RI60 day and late
ones on days 12 to 18. The files label the four columns DMS 465 nm, DMS 405 nm, DLS 465 nm and DLS
405 nm, with the modulation frequencies the Methods give (211 and 330 Hz for DMS, 450 and 270 Hz for
DLS). The stored fiber coordinates lose the signs the Methods give (DV 2.8 for -2.8, DLS AP 0.1 for
-0.1). The authors' code comments call the first column of each pair the control and the second the
signal; the data side with the files: the rewarded nosepoke response computed as labelled is larger
than with the two channels swapped in most sessions (printed above). The response is above that for the
same events shifted by 37 s in most sessions of both regions, so the behavioral events and the
photometry share a clock. The DLS response has the larger area 2 to 10 s after the nosepoke, matching
the prolonged DLS component the Results describe, which is consistent with the region labels; it does
not prove them. In one session (332.393, 2020-07-28) the reward times are MedPC times that do not
coincide with any nosepoke time, which are TDT times, so it has no rewarded nosepokes and no score. The
DLS channels of one session (141.308, 2019-08-09) drop out so often that, with 1 s around each dropout,
99.97% of its samples are left out; that DLS recording is excluded. Some
recordings are shorter than their behavioral session; events outside the recording are dropped. Some
recordings hold dropouts where a raw channel falls near zero, which give |z| above 1000 without the
dropout step.

## Fiber photometry cohort: behavior

Shared tables: per mouse the early and late probe counts, the omission completion time, and per
training session (RI60 or RR20, up to the late probe) the active nosepoke rate, rewards, waits and extra
nosepokes.

In [ ]:
def mixed_anova(df, between, within="probe", dv="shocks", subject="subject"):
    """Two-way mixed ANOVA (one between, one within factor of two levels), complete cases."""
    w = df.pivot_table(index=[subject, between], columns=within, values=dv).dropna().reset_index()
    levels = [c for c in w.columns if c not in (subject, between)]
    a, b = w[levels[0]].to_numpy(), w[levels[1]].to_numpy()
    g = w[between].to_numpy()
    groups = pd.unique(g)
    n, k = len(w), len(groups)
    mean_subj = (a + b) / 2
    grand = mean_subj.mean()
    ss_between = 2 * sum(((mean_subj[g == x].mean() - grand) ** 2) * (g == x).sum() for x in groups)
    ss_subj_err = 2 * sum(((mean_subj[g == x] - mean_subj[g == x].mean()) ** 2).sum() for x in groups)
    d = b - a
    ss_time = n * (d.mean() ** 2) / 2
    ss_inter = sum((g == x).sum() * (d[g == x].mean() - d.mean()) ** 2 for x in groups) / 2
    ss_within_err = sum(((d[g == x] - d[g == x].mean()) ** 2).sum() for x in groups) / 2
    df_b, df_e = k - 1, n - k
    out = {}
    for name, ss, dfn, sse in [(between, ss_between, df_b, ss_subj_err), (within, ss_time, 1, ss_within_err),
                               (f"{between} x {within}", ss_inter, df_b, ss_within_err)]:
        F = (ss / dfn) / (sse / df_e)
        out[name] = {"F": F, "df": f"{dfn},{df_e}", "p": stats.f.sf(F, dfn, df_e)}
    return pd.DataFrame(out).T, w


def fmt(aov):
    return aov.assign(F=aov.F.map("{:.2f}".format), p=aov.p.map("{:.2g}".format)).to_string()


def tukey(values, groups):
    t = pairwise_tukeyhsd(np.asarray(values, float), np.asarray(groups))
    return pd.DataFrame(t._results_table.data[1:], columns=t._results_table.data[0])


fp_long = fp_probes[fp_probes.probe != ""].merge(fp_subj[["subject", "pheno", "sex", "behavior"]], on="subject")
fp_long_sum = fp_long.assign(shocks=fp_long.shocks_sum)
fp_el["schedule"] = fp_el.behavior

om = fp[fp.stage == "omission"].sort_values("start_ts").groupby("subject").asset_id.first()
fp_el["omission_min"] = [seiler.omission_time_min(events[om[s]]["events"]) if s in om.index else np.nan for s in fp_el.index]

late_date = fp_probes[fp_probes.probe == "late"].set_index("subject").date
train = fp[fp.stage.isin(["RI60", "RR20"])].sort_values("start_ts").copy()
train = train[[r.subject not in late_date.index or r.date < late_date[r.subject] for r in train.itertuples()]]
rows = []
for r in train.itertuples():
    ev = events[r.asset_id]["events"]
    pokes = seiler.active_pokes(ev, r.side)
    rm = seiler.reward_measures(ev, r.side)
    rows.append({"subject": r.subject, "date": r.date, "stage": r.stage, "n_rewards": len(seiler.rewards(ev)),
                 "rate": len(pokes) / (seiler.session_length(ev) / 60.0), "wait": rm["wait"], "extra": rm["extra"],
                 "has_ports": rm["has_ports"], "iri": rm["iri"]})
tr = pd.DataFrame(rows).merge(fp_subj[["subject", "pheno", "sex"]], on="subject")
tr["day"] = tr.groupby("subject").cumcount() + 1
per_mouse = tr.groupby("subject").agg(rate=("rate", "mean"), rewards=("n_rewards", "mean"), wait=("wait", "mean"),
                                      extra=("extra", "mean"), sessions=("rate", "size")).join(fp_subj.set_index("subject").pheno)
print("Training sessions used (up to the late probe):", len(tr), "; per mouse", per_mouse.sessions.min(), "to", per_mouse.sessions.max())
print("Sessions without port entries (no wait or extra nosepokes):", int((~tr.has_ports).sum()))
print("Ranges: rate", tr.rate.min().round(2), "to", tr.rate.max().round(1), "per min; rewards", tr.n_rewards.min(), "to", tr.n_rewards.max())
print(per_mouse.groupby("pheno").size().to_string())

### fig1d-ri60-vs-rr20-shocks

> Mice trained on RI60 accept more shocks in the shock probes than mice trained on RR20 (main effect of
> schedule, F1,43 = 6.37, p < 0.05; schedule by time interaction, F1,43 = 4.54, p < 0.05), and RI60 mice
> accept more shocks on the late probe than on the early one (p < 0.01).

In [ ]:
def fig1d(long):
    aov, w = mixed_anova(long, "behavior")
    ri = w[w.behavior == "RI60"]
    t_ = stats.ttest_rel(ri["late"], ri["early"])
    return aov, w, t_

aov, w, t_ = fig1d(fp_long)
print(fmt(aov))
print(w.groupby("behavior")[["early", "late"]].agg(["mean", "std", "size"]).round(2).to_string())
print(f"RI60 late vs early: paired t = {t_.statistic:.2f}, p = {t_.pvalue:.2g}, Bonferroni x2 p = {min(1, 2 * t_.pvalue):.2g}")
rr = w[w.behavior == "RR20"]
print(f"RR20 late vs early: paired t p = {stats.ttest_rel(rr['late'], rr['early']).pvalue:.2g} (n = {len(rr)})")
aov_s, w_s, t_s = fig1d(fp_long_sum)
print("\nVariant, shocks summed over a probe day's files:"); print(fmt(aov_s)); print(
      f"RI60 late vs early Bonferroni p = {min(1, 2 * t_s.pvalue):.2g}")
fig, ax = plt.subplots(figsize=(3.5, 3))
for sched, c in [("RI60", "k"), ("RR20", "grey")]:
    d = w[w.behavior == sched]
    for _, r in d.iterrows():
        ax.plot([0, 1], [r.early, r.late], c, alpha=.3, lw=.8)
    ax.plot([0, 1], [d.early.mean(), d.late.mean()], c, lw=3, label=f"{sched} (n={len(d)})")
ax.set_xticks([0, 1], ["early", "late"]); ax.set_ylabel("shocks"); ax.legend(); plt.tight_layout(); plt.show()
print("Plotted counts span", w[["early", "late"]].min().min(), "to", w[["early", "late"]].max().max())

**Result: not reproduced.** In the 32 RI60 and 5 RR20 mice with both probes, RI60 mice accept
more shocks on average (early 11.38, late 16.31, against 4.80 and 6.60), but the schedule effect has
p = 0.071 (F1,35 = 3.46) and the interaction p = 0.59. The RI60 increase from early to late has paired
p = 0.032, which is 0.063 after the Bonferroni correction, above the claim's 0.01. The dandiset holds 6
RR20 mice, 5 with both probes, where Figure 1D has 9. Summing a probe day's files changes none of this.

### fig1e-omission-ri60-longer

> RI60-trained mice take longer to complete the omission probe than RR20-trained mice (unpaired t-test,
> p < 0.05; n = 20 and 7).

In [ ]:
o = fp_el.dropna(subset=["omission_min"])
print(o.groupby("schedule").omission_min.agg(["mean", "std", "size", "min", "max"]).round(2).to_string())
t1 = stats.ttest_ind(o[o.schedule == "RI60"].omission_min, o[o.schedule == "RR20"].omission_min)
print(f"Unpaired t-test RI60 vs RR20: t = {t1.statistic:.2f}, p = {t1.pvalue:.3g}")
print(f"Welch variant: p = {stats.ttest_ind(o[o.schedule == 'RI60'].omission_min, o[o.schedule == 'RR20'].omission_min, equal_var=False).pvalue:.3g}")
print("Mice with fewer than 50 rewards (60 min):", (o.omission_min >= 60).groupby(o.schedule).sum().to_dict())
print("RI60 omission times by phenotype:", o[o.schedule == "RI60"].groupby("pheno").omission_min.agg(["mean", "size"]).round(1).to_dict("index"))

**Result: partially reproduced.** RI60 mice take 33.97 min to complete the omission probe
(n = 19) and RR20 mice 17.72 min (n = 4); the unpaired t-test gives p = 0.070, so the direction holds
without significance. The RR20 times vary little (SD 1.05), and a Welch test gives p = 0.0005. The
dandiset holds 4 RR20 omission probes where the paper has 7, and 19 RI60 ones where it has 20.

### fig1f-omission-shock-no-correlation

> Among RI60 mice, shocks received on the late probe and omission completion time are not significantly
> correlated (r = 0.42, ns).

In [ ]:
o1 = fp_el[(fp_el.schedule == "RI60")].dropna(subset=["omission_min", "late"])
r1 = stats.pearsonr(o1.late, o1.omission_min)
print(f"Pearson r = {r1.statistic:.3f}, p = {r1.pvalue:.3g}, n = {len(o1)}")
o1e = fp_el[(fp_el.schedule == "RI60")].dropna(subset=["omission_min", "early"])
print(f"Variant, early probe: r = {stats.pearsonr(o1e.early, o1e.omission_min).statistic:.3f}, p = {stats.pearsonr(o1e.early, o1e.omission_min).pvalue:.3g}, n = {len(o1e)}")
print(f"Spearman, late probe: rho = {stats.spearmanr(o1.late, o1.omission_min).statistic:.3f}, p = {stats.spearmanr(o1.late, o1.omission_min).pvalue:.3g}")

**Result: reproduced.** Late probe shocks and omission time are not significantly correlated
(r = 0.335, p = 0.175, n = 18; the paper gives r = 0.42, ns). With the early probe the correlation is
significant (r = 0.592, p = 0.0076, n = 19).

### fig2a-phenotype-rule

> RI60 mice are split by their shock probe counts into 9 PR, 9 DPR and 18 PS mice, by the rule in the
> Methods (DPR when the count rises by more than 85%; otherwise PR when the early count exceeds 13).

In [ ]:
ri = fp_el[fp_el.schedule == "RI60"].copy()
print("Stored labels of RI60 mice:", ri.pheno.value_counts().to_dict())
both = ri.dropna(subset=["early", "late"]).copy()
both["rule"] = [seiler.classify(e, l) for e, l in zip(both.early, both.late)]
both["change_pct"] = 100 * (both.late - both.early) / both.early
print(f"Mice with both probes: {len(both)}; rule agrees with the stored label for {(both.rule == both.pheno).sum()}")
print(pd.crosstab(both.pheno, both.rule).to_string())
print("\nDisagreements:")
print(both[both.rule != both.pheno][["early", "late", "change_pct", "pheno", "rule"]].round(1).to_string())
print(f"\nMedian early count of the RI60 mice: {ri.early.median():.1f} (Methods: median split at 13)")
print(f"75th percentile of the percent change: {both.change_pct.quantile(.75):.1f} (Methods: top quartile, above 85%)")
both["rule_ge13"] = ["DPR" if 100 * (l - e) / e > 85 else ("PR" if e >= 13 else "PS") for e, l in zip(both.early, both.late)]
print("Variant, PR when early >= 13: agrees for", int((both.rule_ge13 == both.pheno).sum()), "of", len(both))
bs = seiler.early_late(fp_probes, "shocks_sum").reindex(both.index)
both["rule_sum"] = [seiler.classify(e, l) for e, l in zip(bs.early, bs.late)]
print("Variant, counts summed over a day's files: agrees for", int((both.rule_sum == both.pheno).sum()), "of", len(both))
one = ri[ri[["early", "late"]].isna().any(axis=1)]
print("\nMice with one probe, label and available count:")
print(one[["early", "late", "pheno"]].to_string())

**Result: partially reproduced.** The stored labels count 9 PR, 9 DPR and 18 PS, as in Figure 2A.
The Methods rule gives the stored label for 31 of the 32 mice with both probes; mouse 048.392 has 13
shocks on its early probe and a 38.5% rise, so the rule ("more than 13") makes it PS, while it is stored
as PR. With PR at 13 or more all 32 agree. The 85% cut matches the top quartile of change here (75th
percentile 87.7%), but the median early count is 9.0, not 13, so the "median split" at 13 is not a median
of these mice.

### fig2b-only-dpr-increase

> There is an interaction of phenotype and probe on shocks received (p < 0.0001), and only DPR mice
> receive significantly more shocks on the late probe than the early one.

In [ ]:
ri_long = fp_long[fp_long.behavior == "RI60"]
aov2, w2 = mixed_anova(ri_long, "pheno")
print(fmt(aov2))
print(w2.groupby("pheno")[["early", "late"]].agg(["mean", "std", "size"]).round(2).to_string())
for ph_ in ["PR", "DPR", "PS"]:
    d = w2[w2.pheno == ph_]
    p = stats.ttest_rel(d.late, d.early).pvalue
    print(f"{ph_}: late - early mean {np.mean(d.late - d.early):.2f}, paired t p = {p:.2g}, Bonferroni x3 p = {min(1, 3 * p):.2g}, n = {len(d)}")

**Result: reproduced.** The phenotype by probe interaction has F2,29 = 26.37, p below 0.0001. DPR
mice rise from 11.56 to 32.11 shocks (Bonferroni p = 0.00041); PR (23.50 to 20.50) and PS (7.00 to 6.47)
do not change (corrected p = 1 for both).

### fig2c-nosepoke-rate

> During RI60 training PR and DPR mice nosepoke at higher rates than PS mice (main effect of phenotype,
> p < 0.0001).

In [ ]:
pm = per_mouse[per_mouse.pheno.isin(["PR", "DPR", "PS"])]
print(pm.groupby("pheno").rate.agg(["mean", "std", "size", "min", "max"]).round(2).to_string())
print(f"One-way ANOVA: p = {stats.f_oneway(*[pm[pm.pheno == g].rate for g in ['PR', 'DPR', 'PS']]).pvalue:.3g}")
print(tukey(pm.rate, pm.pheno).to_string())
trr = tr[tr.pheno.isin(["PR", "DPR", "PS"])]
m1 = smf.mixedlm("rate ~ C(pheno) + day", trr, groups=trr.subject).fit(reml=False)
m0 = smf.mixedlm("rate ~ day", trr, groups=trr.subject).fit(reml=False)
lr = 2 * (m1.llf - m0.llf)
print(f"Variant, mixed model over {len(trr)} sessions: phenotype likelihood ratio {lr:.1f}, p = {stats.chi2.sf(lr, 2):.2g}")
print("Mean rate by sex within PS (sex is unbalanced across phenotypes):", pm.join(fp_subj.set_index("subject").sex).groupby(["pheno", "sex"]).rate.agg(["mean", "size"]).round(2).to_dict("index"))

**Result: reproduced.** Over RI60 training PR mice make 14.20 and DPR mice 12.79 active nosepokes
per minute, PS mice 5.24 (one-way ANOVA p = 4.3e-06); Tukey gives PR > PS with p below 0.0001 and
DPR > PS with p = 0.0002. A mixed model over 493 sessions agrees (p = 1.8e-06). Sex is confounded with
phenotype: 17 of 18 PS mice are female and 8 of 9 PR mice male.

### fig2d-omission-pr-vs-ps

> PR mice take longer than PS mice to complete the omission probe (PR 51.50 +- 17, PS 27.92 +- 13.65;
> one-way ANOVA F2,18 = 4.36, p < 0.05; Tukey PR vs PS p < 0.05).

In [ ]:
o2 = fp_el[(fp_el.schedule == "RI60")].dropna(subset=["omission_min"])
print(o2.groupby("pheno").omission_min.agg(["mean", "std", "size", "min", "max"]).round(2).to_string(), "(minutes; paper n PR 4, DPR 5, PS 11)")
an = stats.f_oneway(*[o2[o2.pheno == g].omission_min for g in ["PR", "DPR", "PS"]])
print(f"One-way ANOVA: F = {an.statistic:.2f}, p = {an.pvalue:.3g}, df = 2,{len(o2) - 3}")
print(tukey(o2.omission_min, o2.pheno).to_string())
print("Values per mouse (min):", o2.sort_values("pheno")[["pheno", "omission_min"]].round(1).to_dict("records"))

**Result: reproduced.** PR mice take 51.65 min (n = 4) and PS mice 28.06 min (n = 10), both
within the stated SDs of 51.50 and 27.92; one-way ANOVA F2,16 = 3.85, p = 0.043 (paper F2,18 = 4.36);
Tukey PR vs PS p = 0.037. Three of the four PR mice reach the 60 min limit. The values fit minutes, not
the seconds the Results print.

### fig2k-ps-wait-longer

> After a reward PS mice wait longer before nosepoking again (38 +- 15 s) than PR (21 +- 5 s) and DPR
> (22 +- 4 s) mice (one-way ANOVA p < 0.01; Tukey p < 0.01 for each).

In [ ]:
print(pm.groupby("pheno").wait.agg(["mean", "std", "size", "min", "max"]).round(2).to_string())
print(f"One-way ANOVA: p = {stats.f_oneway(*[pm[pm.pheno == g].wait.dropna() for g in ['PR', 'DPR', 'PS']]).pvalue:.3g}")
print(tukey(pm.wait, pm.pheno).to_string())
med = tr[tr.pheno.isin(["PR", "DPR", "PS"])].groupby(["subject", "pheno"]).wait.median().reset_index()
print("Variant, median over sessions per mouse:", med.groupby("pheno").wait.mean().round(2).to_dict())

**Result: partially reproduced.** PS mice wait longest after the rewarded port entry (53.94 s,
SD 44.67), against PR 21.05 s and DPR 21.86 s, which are within tolerance; the PS mean is above its
tolerance (23 to 53 s), pulled up by a few mice (maximum 171 s; with each mouse's median session the PS
mean is 32.2 s). The ANOVA has p = 0.017; Tukey PS > PR has p = 0.045 and PS > DPR p = 0.051, so one of
the two pairwise tests fails.

### fig2l-extra-nosepokes

> PS mice make fewer extra nosepokes between a rewarded nosepoke and the rewarded port entry than PR
> mice (PS 1.28 +- 0.85, PR 3.23 +- 1.65, DPR 2.1 +- 2.63; Tukey PS vs PR p < 0.05).

In [ ]:
print(pm.groupby("pheno").extra.agg(["mean", "std", "size", "min", "max"]).round(2).to_string())
print(f"One-way ANOVA: p = {stats.f_oneway(*[pm[pm.pheno == g].extra.dropna() for g in ['PR', 'DPR', 'PS']]).pvalue:.3g}")
print(tukey(pm.extra, pm.pheno).to_string())

**Result: partially reproduced.** PS mice make the fewest extra nosepokes (1.61 per reward,
PR 3.33, DPR 2.28), all within the stated SDs, but Tukey PS vs PR gives p = 0.117 (ANOVA p = 0.139).

### fig2m-inter-reward-interval

> PS mice have longer inter-reward intervals (79.77 +- 26.54 s) than PR (71.2 +- 32.3 s) and DPR
> (69.58 +- 35.18 s) mice (Kolmogorov-Smirnov, p < 0.0001 for each).

In [ ]:
iri = {g: np.concatenate(tr[(tr.pheno == g) & (tr.stage == "RI60")].iri.tolist()) for g in ["PR", "DPR", "PS"]}
for g, v in iri.items():
    print(f"{g}: mean {v.mean():.2f} s, SD {v.std(ddof=1):.2f}, median {np.median(v):.1f}, n intervals {len(v)}, range {v.min():.1f} to {v.max():.0f}")
for g in ["PR", "DPR"]:
    ks = stats.ks_2samp(iri["PS"], iri[g])
    print(f"KS PS vs {g}: D = {ks.statistic:.3f}, p = {ks.pvalue:.2g}")
mouse_iri = tr[tr.stage == "RI60"].groupby(["subject", "pheno"]).iri.apply(lambda x: np.concatenate(x.tolist()).mean()).reset_index()
print("Variant, per mouse mean interval:", mouse_iri.groupby("pheno").iri.agg(["mean", "size"]).round(1).to_dict("index"),
      f"; ANOVA p = {stats.f_oneway(*[mouse_iri[mouse_iri.pheno == g].iri for g in ['PR', 'DPR', 'PS']]).pvalue:.3g}")
fig, ax = plt.subplots(figsize=(4, 3))
bins = np.arange(0, 301, 10)
for g, c in [("PR", "k"), ("DPR", "m"), ("PS", "c")]:
    ax.hist(iri[g], bins, histtype="step", density=True, color=c, label=g)
ax.set_xlabel("inter-reward interval (s)"); ax.legend(); plt.tight_layout(); plt.show()
print("Intervals above 300 s (outside the axes):", {g: int((v > 300).sum()) for g, v in iri.items()})

**Result: reproduced.** Pooled RI60 inter-reward intervals average 81.75 s for PS, 71.96 s for PR
and 69.57 s for DPR, close to the paper's 79.77, 71.2 and 69.58 s; KS tests give p = 5e-30 (PS vs PR)
and 3.3e-33 (PS vs DPR). With one mean per mouse the ANOVA gives p = 0.033.

### fig2n-rewards-per-session

> Averaged over RI60 training, PS mice earn fewer rewards per session (39.91 +- 7.39) than DPR mice
> (46.56 +- 4.49; PR 45.33 +- 4.33); one-way ANOVA p < 0.05, Tukey PS vs DPR p < 0.05.

In [ ]:
print(pm.groupby("pheno").rewards.agg(["mean", "std", "size", "min", "max"]).round(2).to_string())
an = stats.f_oneway(*[pm[pm.pheno == g].rewards for g in ["PR", "DPR", "PS"]])
print(f"One-way ANOVA: F = {an.statistic:.2f}, p = {an.pvalue:.3g}")
print(tukey(pm.rewards, pm.pheno).to_string())
allri = tr[tr.stage == "RI60"].merge(fp[["subject", "date"]].drop_duplicates(), on=["subject", "date"])
print("Sessions with 0 rewards:", int((tr.n_rewards == 0).sum()))

**Result: reproduced.** PS mice earn 38.95 rewards per RI60 session, DPR 44.98 and PR 44.66, all
within tolerance of 39.91, 46.56 and 45.33; ANOVA F = 4.35, p = 0.021; Tukey PS vs DPR p = 0.048.

### fig2o-escalation-timing

> PR mice escalate their nosepoking as soon as they enter RI30, while DPR mice escalate later, during
> RI60.

In [ ]:
ri30 = []
for r in fp[fp.stage == "RI30"].itertuples():
    ev = events[r.asset_id]["events"]
    ri30.append({"subject": r.subject, "rate": len(seiler.active_pokes(ev, r.side)) / (seiler.session_length(ev) / 60.0)})
ri30 = pd.DataFrame(ri30).groupby("subject").rate.mean().to_frame("ri30_rate").join(fp_subj.set_index("subject").pheno)
slopes = tr[tr.stage == "RI60"].groupby("subject").apply(lambda d: np.polyfit(d.day, d.rate, 1)[0] if len(d) > 2 else np.nan).rename("ri60_slope")
esc = ri30.join(slopes, how="outer")
esc["pheno"] = esc.index.map(fp_subj.set_index("subject").pheno)
esc = esc[esc.pheno.isin(["PR", "DPR", "PS"])]
print(esc.groupby("pheno").agg(ri30_rate_mean=("ri30_rate", "mean"), ri30_n=("ri30_rate", "count"),
                               ri60_slope_median=("ri60_slope", "median"), ri60_slope_mean=("ri60_slope", "mean"), ri60_n=("ri60_slope", "count")).round(3).to_string())
for g in ["DPR", "PS"]:
    p = stats.mannwhitneyu(esc[esc.pheno == "PR"].ri30_rate.dropna(), esc[esc.pheno == g].ri30_rate.dropna()).pvalue
    print(f"RI30 rate PR vs {g}: Mann-Whitney p = {p:.3g}")
p = stats.mannwhitneyu(esc[esc.pheno == "DPR"].ri60_slope.dropna(), esc[esc.pheno == "PR"].ri60_slope.dropna()).pvalue
print(f"RI60 slope DPR vs PR: Mann-Whitney p = {p:.3g}")
fig, ax = plt.subplots(figsize=(5, 3))
for g, c in [("PR", "k"), ("DPR", "m"), ("PS", "c")]:
    d = tr[(tr.pheno == g) & (tr.stage == "RI60")].groupby("day").rate.mean()
    ax.plot(d.index, d.values, c, label=g)
ax.set_xlabel("RI60 day"); ax.set_ylabel("active nosepokes / min"); ax.legend(); plt.tight_layout(); plt.show()
print("Plotted day means span", tr[tr.stage == "RI60"].groupby(["pheno", "day"]).rate.mean().min().round(2), "to", tr[tr.stage == "RI60"].groupby(["pheno", "day"]).rate.mean().max().round(2))

**Result: partially reproduced.** In RI30 PR mice nosepoke at 4.85 per minute, above PS (2.22,
p = 0.0063) but not significantly above DPR (3.03, p = 0.114), so part (i) fails. Over RI60 the median
slope of DPR mice (0.809 nosepokes/min per day) is above that of PR mice (0.494), so part (ii) holds,
though this difference has p = 0.289.

### figs1i-ri60-more-rewards

> RI60-trained mice earn more rewards per training session than RR20-trained mice (unpaired t-test,
> p < 0.0001).

In [ ]:
pmr = per_mouse.assign(schedule=np.where(per_mouse.pheno == "RR20", "RR20", "RI60"))
print(pmr.groupby("schedule").rewards.agg(["mean", "std", "size", "min", "max"]).round(2).to_string())
t2 = stats.ttest_ind(pmr[pmr.schedule == "RI60"].rewards, pmr[pmr.schedule == "RR20"].rewards)
print(f"Unpaired t-test: t = {t2.statistic:.2f}, p = {t2.pvalue:.2g}")
pmr["rate"] = per_mouse.rate
print("Active nosepokes per minute by schedule (Fig S1G, similar rates):", pmr.groupby("schedule").rate.agg(["mean", "std"]).round(2).to_dict("index"))

**Result: reproduced.** RI60 mice earn 41.89 rewards per training session and RR20 mice 15.38
(t-test p = 1.4e-11, below 0.0001).

### figs2b-males-more-shocks

> Male mice accept more shocks than female mice in the shock probes (main effect of sex, F1,35 = 26.19,
> p < 0.0001).

In [ ]:
aov3, w3 = mixed_anova(ri_long, "sex")
print(fmt(aov3))
print(w3.groupby("sex")[["early", "late"]].agg(["mean", "std", "size"]).round(2).to_string())
print("Sex by phenotype (RI60, all mice):", pd.crosstab(fp_subj[fp_subj.behavior == "RI60"].pheno, fp_subj[fp_subj.behavior == "RI60"].sex).to_dict())

**Result: reproduced.** Males accept more shocks than females (early 16.58 and late 23.25,
against 8.25 and 12.15; sex effect F1,30 = 9.81, p = 0.0039), but not with the paper's p below 0.0001.
Sex and phenotype are confounded in this cohort (PR 8 males of 9, PS 17 females of 18).

## Fiber photometry cohort: dopamine axon signals

### fig3e-ri30-dms-peaks

> Peaks in DMS dopamine axon activity at rewarded nosepokes emerge during RI30 in PR mice, and more
> slowly across RI60 in DPR mice.

Blocked: the dandiset holds photometry only for RI60 and RR20 sessions (and one session labelled as a
probe test), not for RI30, so the emergence of peaks during RI30 cannot be tested. The early and late
RI60 PSTHs are shown in the data check above.

### fig3f-dms-score-phenotype

> The DMS rewarded-unrewarded peak score differs by phenotype (F2,33 = 8.16, p < 0.01) and changes
> across training (p < 0.001).

In [ ]:
# One score per mouse and stage: the mean over its sessions in that stage (one mouse has two early sessions).
sc = ph[ph.pheno.isin(["PR", "DPR", "PS"])].groupby(["subject", "pheno", "fp_stage"])[
    ["DMS_score", "DLS_score", "DMS_score_nomask", "DLS_score_nomask"]].mean().reset_index()
print("Mice per phenotype and stage with a DMS score:")
print(sc.dropna(subset=["DMS_score"]).groupby(["pheno", "fp_stage"]).size().unstack().to_string())
print("Mice per phenotype and stage with a DLS score:")
print(sc.dropna(subset=["DLS_score"]).groupby(["pheno", "fp_stage"]).size().unstack().to_string())


def phenotype_model(df, col):
    d = df.dropna(subset=[col]).rename(columns={col: "y"})
    full = smf.mixedlm("y ~ C(pheno) * C(fp_stage)", d, groups=d.subject).fit(reml=False)
    no_ph = smf.mixedlm("y ~ C(fp_stage)", d, groups=d.subject).fit(reml=False)
    no_stage = smf.mixedlm("y ~ C(pheno)", d, groups=d.subject).fit(reml=False)
    lr_p = 2 * (full.llf - no_ph.llf)
    lr_s = 2 * (full.llf - no_stage.llf)
    out = {"n_obs": len(d), "n_mice": d.subject.nunique(),
           "phenotype_LR": lr_p, "phenotype_p": stats.chi2.sf(lr_p, 4),
           "stage_LR": lr_s, "stage_p": stats.chi2.sf(lr_s, 3)}
    for st in ["early", "late"]:
        s = d[d.fp_stage == st]
        out[f"anova_{st}_p"] = stats.f_oneway(*[s[s.pheno == g].y for g in ["PR", "DPR", "PS"] if (s.pheno == g).sum() > 1]).pvalue
    return out, d.groupby(["pheno", "fp_stage"]).y.agg(["mean", "std", "size"])

In [ ]:
res3f, tab3f = phenotype_model(sc, "DMS_score")
print(tab3f.round(3).unstack().to_string())
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res3f.items()})
print("Means over both stages:", sc.groupby("pheno").DMS_score.mean().round(3).to_dict())
d = sc.dropna(subset=["DMS_score"])
for st in ["early", "late"]:
    s = d[d.fp_stage == st]
    print(st, "Tukey:"); print(tukey(s.DMS_score, s.pheno).to_string())
d2 = d.pivot(index="subject", columns="fp_stage", values="DMS_score").dropna()
print(f"Early vs late, paired over {len(d2)} mice: mean change {np.mean(d2.late - d2.early):.3f}, paired t p = {stats.ttest_rel(d2.late, d2.early).pvalue:.3g}")
resn, tabn = phenotype_model(sc, "DMS_score_nomask")
print("Variant, the authors' processing without the dropout step:", {k: round(v, 4) for k, v in resn.items() if "p" in k})
print("Per-mouse scores span", round(d.DMS_score.min(), 2), "to", round(d.DMS_score.max(), 2))
fig, ax = plt.subplots(figsize=(4, 3))
for i, g in enumerate(["PR", "DPR", "PS"]):
    for j, st in enumerate(["early", "late"]):
        v = d[(d.pheno == g) & (d.fp_stage == st)].DMS_score
        ax.scatter(np.full(len(v), i * 3 + j) + np.random.default_rng(0).uniform(-.15, .15, len(v)), v, s=10, c="k" if st == "early" else "r")
        ax.plot([i * 3 + j - .3, i * 3 + j + .3], [v.mean()] * 2, "b")
ax.set_xticks([0.5, 3.5, 6.5], ["PR", "DPR", "PS"]); ax.set_ylabel("DMS score (z); black early, red late"); plt.tight_layout(); plt.show()

**Result: not reproduced.** The DMS score is ordered as the paper describes (mean PR 2.35, DPR
1.66, PS 1.05 z), but the phenotype effect has likelihood ratio p = 0.100, and neither stage alone is
significant (one-way ANOVA p = 0.124 early, 0.270 late; no Tukey pair below 0.05). The score rises from
early to late in the 17 mice with both (mean 0.49, paired p = 0.046; model stage p = 0.088). With the
authors' processing without the dropout step the phenotype p is 0.389. RI30 sessions, part of the paper's
test, are not in the dandiset.

### fig3g-dms-score-shock-correlation

> The DMS rewarded-unrewarded peak score correlates with the shocks a mouse receives in the shock probes
> (r = 0.53, p < 0.0001).

In [ ]:
pairs = sc.merge(fp_probes[fp_probes.probe != ""][["subject", "probe", "shocks"]].rename(columns={"probe": "fp_stage"}), on=["subject", "fp_stage"])
def corr(df, x, y="shocks"):
    d = df.dropna(subset=[x, y])
    r = stats.pearsonr(d[x], d[y])
    return f"r = {r.statistic:.3f}, p = {r.pvalue:.2g}, n = {len(d)}"
print("(a) pooled early and late:", corr(pairs, "DMS_score"))
print("(b) late only:            ", corr(pairs[pairs.fp_stage == "late"], "DMS_score"))
print("(c) early only:           ", corr(pairs[pairs.fp_stage == "early"], "DMS_score"))
print("Spearman, pooled:", stats.spearmanr(pairs.dropna(subset=["DMS_score"]).DMS_score, pairs.dropna(subset=["DMS_score"]).shocks))
print("Variant without the dropout step, pooled:", corr(pairs, "DMS_score_nomask"))
rrp = ph[ph.pheno == "RR20"].groupby(["subject", "fp_stage"]).DMS_score.mean().reset_index().merge(
    fp_probes[fp_probes.probe != ""][["subject", "probe", "shocks"]].rename(columns={"probe": "fp_stage"}), on=["subject", "fp_stage"])
print("Variant with RR20 mice added, pooled:", corr(pd.concat([pairs, rrp]), "DMS_score"))
print("Variant, each mouse's mean score against its mean shocks:", corr(pairs.groupby("subject")[["DMS_score", "shocks"]].mean().reset_index(), "DMS_score"))
fig, ax = plt.subplots(figsize=(3.5, 3))
for st, c in [("early", "k"), ("late", "r")]:
    s = pairs[pairs.fp_stage == st]
    ax.scatter(s.DMS_score, s.shocks, c=c, s=12, label=st)
ax.set_xlabel("DMS score (z)"); ax.set_ylabel("shocks"); ax.legend(); plt.tight_layout(); plt.show()
print("Plotted points span score", round(pairs.DMS_score.min(), 2), "to", round(pairs.DMS_score.max(), 2), "and shocks", pairs.shocks.min(), "to", pairs.shocks.max())

**Result: partially reproduced.** Pooling early and late pairs (reading a) gives r = 0.170,
p = 0.24, n = 49; late pairs (b) r = 0.014, p = 0.95; early pairs (c) r = 0.375, p = 0.045, n = 29. Only
reading (c) is positive and significant, and its r is below the tolerance around 0.53.

### fig3h-dms-score-omission-no-correlation

> The DMS rewarded-unrewarded peak score does not correlate with omission completion time (r = 0.1,
> ns).

In [ ]:
oms = sc.merge(fp_el[["omission_min"]].reset_index(), on="subject")
print("Late score vs omission time:", corr(oms[oms.fp_stage == "late"], "DMS_score", "omission_min"))
print("Variant, pooled early and late:", corr(oms, "DMS_score", "omission_min"))
print("Variant, early score:", corr(oms[oms.fp_stage == "early"], "DMS_score", "omission_min"))

**Result: reproduced.** The late DMS score does not correlate with omission time (r = 0.014,
p = 0.96, n = 12); pooled early and late r = -0.050, p = 0.80.

### fig4f-dls-score-no-phenotype

> The DLS rewarded-unrewarded peak score has no significant effect of phenotype.

In [ ]:
res4f, tab4f = phenotype_model(sc, "DLS_score")
print(tab4f.round(3).unstack().to_string())
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res4f.items()})
resn4, _ = phenotype_model(sc, "DLS_score_nomask")
print("Variant, without the dropout step:", {k: round(v, 4) for k, v in resn4.items() if "p" in k})

**Result: reproduced.** The DLS score has no phenotype effect (likelihood ratio p = 0.48; per stage
p = 0.61 and 0.47). It changes with stage (p = 0.034), as the paper also reports for DLS.

### fig4g-dls-score-shock-no-correlation

> The DLS rewarded-unrewarded peak score does not correlate with shocks received (r = -0.01, ns).

In [ ]:
print("(a) pooled early and late:", corr(pairs, "DLS_score"))
print("Variant, late only:", corr(pairs[pairs.fp_stage == "late"], "DLS_score"))
print("Variant, early only:", corr(pairs[pairs.fp_stage == "early"], "DLS_score"))
print("Variant without the dropout step, pooled:", corr(pairs, "DLS_score_nomask"))

**Result: not reproduced.** The pooled DLS score correlates with shocks received (r = 0.326,
p = 0.024, n = 48), where the paper finds none (r = -0.01). Late pairs alone give r = 0.374, p = 0.10, and
early pairs r = 0.103, p = 0.60. Without the dropout step the pooled r is 0.113, p = 0.45, so the result
depends on how the artifacts are handled.

### fig4h-dls-score-omission-no-correlation

> The DLS rewarded-unrewarded peak score does not correlate with omission completion time (r = -0.21,
> ns).

In [ ]:
print("Late score vs omission time:", corr(oms[oms.fp_stage == "late"], "DLS_score", "omission_min"))
print("Variant, pooled early and late:", corr(oms, "DLS_score", "omission_min"))

**Result: reproduced.** r = 0.311, p = 0.32, n = 12 (paper r = -0.21, ns).

## Optogenetic cohorts

Optogenetic mice carry no phenotype in the dandiset; they are classified here with the Methods rule
from their early and late probes.

In [ ]:
def opto_tables(experiment):
    s = sessions[sessions.subject_experiment == experiment]
    probes = seiler.probe_table(s, events, training_stages=("RI60",))
    el = seiler.early_late(probes).join(s.drop_duplicates("subject").set_index("subject")[["subject_arm", "sex"]], how="right")
    el["pheno"] = [seiler.classify(e, l) if pd.notna(e) and pd.notna(l) else None for e, l in zip(el.early, el.late)]
    omf = s[s.stage == "omission"].sort_values("start_ts").groupby("subject").asset_id.first()
    el["omission_min"] = [seiler.omission_time_min(events[omf[x]]["events"]) if x in omf.index else np.nan for x in el.index]
    el["n_ri60"] = s[s.stage == "RI60"].drop_duplicates(["subject", "start"]).groupby("subject").size().reindex(el.index).fillna(0).astype(int)
    el["did_not_learn"] = el.index.map(opto_subj.set_index("subject").did_not_learn)
    return s, probes, el

opto = {e: opto_tables(e) for e in ["Opto-DMS-Excitatory", "Opto-DLS-Excitatory", "Opto-DMS-Inhibitory"]}
for e, (s, probes, el) in opto.items():
    print(f"== {e}: {len(el)} mice")
    print("Mice per arm and sex:", el.groupby(["subject_arm", "sex"]).size().unstack(fill_value=0).to_dict("index"))
    print("Probe days per mouse:", probes.groupby("subject").size().value_counts().sort_index().to_dict(),
          "; days with two files holding shocks:", int((probes.n_with_shocks > 1).sum()))
    print("Training days before early / late probe:", probes[probes.probe == "early"].train_days_before.describe()[["min", "50%", "max"]].to_dict(),
          probes[probes.probe == "late"].train_days_before.describe()[["min", "50%", "max"]].to_dict())
    print("Mice with both probes:", int(el[["early", "late"]].notna().all(axis=1).sum()), "; with an omission probe:", int(el.omission_min.notna().sum()))
    print("Mice missing a probe:", el[el[["early", "late"]].isna().any(axis=1)][["subject_arm", "early", "late", "n_ri60", "did_not_learn"]].to_dict("index"))

### fig5f-dms-chr2-escalation

> Mice receiving DMS dopamine terminal stimulation on rewarded nosepokes (ChR2) escalate their nosepoking
> faster than EYFP and ChR2 Scrambled controls during FR1 and RI30.

In [ ]:
def escalation(experiment, ref="ChR2"):
    s = sessions[(sessions.subject_experiment == experiment) & sessions.stage.isin(["FR1", "RI30"])].sort_values("start_ts")
    s = s[~((s.arm == "Unknown") & s.duplicated(["subject", "start"], keep=False))].drop_duplicates(["subject", "start"])
    rows = []
    for r in s.itertuples():
        ev = events[r.asset_id]["events"]
        if isinstance(r.side, str):
            pokes = seiler.active_pokes(ev, r.side)
        else:  # FR1 on both nosepokes
            pokes = np.concatenate([seiler.active_pokes(ev, "left"), seiler.active_pokes(ev, "right")])
        rows.append({"subject": r.subject, "arm": r.subject_arm, "stage": r.stage, "rate": len(pokes) / (seiler.session_length(ev) / 60.0)})
    d = pd.DataFrame(rows)
    d["idx"] = d.groupby(["subject", "stage"]).cumcount() + 1
    out = {}
    for st in ["FR1", "RI30"]:
        x = d[d.stage == st]
        fit = smf.ols(f"rate ~ idx * C(arm, Treatment('{ref}'))", x).fit()
        slopes = {a: np.polyfit(x[x.arm == a].idx, x[x.arm == a].rate, 1)[0] for a in sorted(x.arm.unique())}
        inter = {k.split("T.")[1].rstrip("]"): (fit.params[k], fit.pvalues[k]) for k in fit.params.index if k.startswith("idx:")}
        others = [a for a in sorted(x.arm.unique()) if a != ref]
        f2 = smf.ols(f"rate ~ idx * C(arm, Treatment('{others[0]}'))", x).fit()
        k2 = [k for k in f2.params.index if k.startswith("idx:") and others[1] in k][0]
        out[st] = {"slopes": slopes, "interaction_vs_ref": inter, "sessions": len(x), "mice": x.subject.nunique(),
                   "other_pair": (f"{others[1]} vs {others[0]}", f2.params[k2], f2.pvalues[k2]),
                   "sessions_per_mouse": (x.groupby("subject").size().min(), x.groupby("subject").size().max())}
    return d, out

In [ ]:
esc5, res5 = escalation("Opto-DMS-Excitatory")
for st, r in res5.items():
    print(st, f"{r['sessions']} sessions from {r['mice']} mice ({r['sessions_per_mouse'][0]} to {r['sessions_per_mouse'][1]} per mouse)")
    print("  slope (nosepokes/min per session):", {k: round(v, 3) for k, v in r["slopes"].items()})
    print("  arm x index interaction against ChR2 (difference in slope, p):", {k: (round(v[0], 3), round(v[1], 4)) for k, v in r["interaction_vs_ref"].items()})
print("Mean rate by stage and arm:", esc5.groupby(["stage", "arm"]).rate.mean().round(2).to_dict())
per_mouse_slope = esc5.groupby(["stage", "arm", "subject"]).apply(lambda x: np.polyfit(x.idx, x.rate, 1)[0] if len(x) > 1 else np.nan).rename("slope").reset_index()
print("Variant, per-mouse slopes (median):", per_mouse_slope.groupby(["stage", "arm"]).slope.median().round(3).to_dict())
for st in ["FR1", "RI30"]:
    x = per_mouse_slope[per_mouse_slope.stage == st]
    for a in ["EYFP", "ChR2Scrambled"]:
        print(f"  {st} ChR2 vs {a}, Mann-Whitney on per-mouse slopes p = {stats.mannwhitneyu(x[x.arm == 'ChR2'].slope.dropna(), x[x.arm == a].slope.dropna()).pvalue:.3g}")

**Result: not reproduced.** In the pooled fit of the criterion no ChR2 slope differs from a
control slope (FR1 interaction p = 0.63 and 0.98; RI30 p = 0.82 and 0.49), and the pooled FR1 slope of
ChR2 mice (0.011) is not above the controls'. Per-mouse slopes tell a different story for FR1: the ChR2
median is 0.657 nosepokes/min per session, above EYFP (-0.021, p = 0.015) and ChR2 Scrambled (0.275,
p = 0.021); in RI30 they do not differ (p = 0.6 and 1). ChR2 mice nosepoke more during RI30 (mean 7.75
per minute against 4.01 and 4.21). The pooled fit mixes mice with 1 to 13 FR1 sessions, so the two
approaches disagree.

### fig5g-dms-chr2-more-shocks-early

> On the first shock probe ChR2 mice accept more shocks than EYFP mice (interaction of time and
> manipulation, F2,35 = 3.65, p < 0.05; Tukey p < 0.05); the difference fades by the second probe.

In [ ]:
_, p5, el5 = opto["Opto-DMS-Excitatory"]
long5 = p5[p5.probe != ""].merge(el5[["subject_arm"]].reset_index(), on="subject")
aov5, w5 = mixed_anova(long5, "subject_arm")
print(fmt(aov5))
print(w5.groupby("subject_arm")[["early", "late"]].agg(["mean", "std", "size"]).round(2).to_string())
print("First probe, Tukey:"); print(tukey(w5.early, w5.subject_arm).to_string())
print("Second probe, Tukey:"); print(tukey(w5.late, w5.subject_arm).to_string())
e1 = el5.dropna(subset=["early"])
print("Variant, all mice with a first probe:", e1.groupby("subject_arm").early.agg(["mean", "size"]).round(2).to_dict("index"))
print(tukey(e1.early, e1.subject_arm).to_string())

**Result: not reproduced.** On the first probe ChR2 mice accept 28.69 shocks, EYFP 21.86 and
ChR2 Scrambled 20.67, the paper's order, but Tukey ChR2 vs EYFP gives p = 0.51 and the group by probe
interaction p = 0.16 (F2,29 = 1.95). The dandiset holds 33 DMS excitatory mice (7 EYFP), where the paper's
statistics imply 38 to 39.

### fig5h-dms-chr2-mostly-pr

> ChR2 mice are very likely to be classified PR, while EYFP and ChR2 Scrambled mice are distributed across
> phenotypes as expected.

In [ ]:
c5 = el5.dropna(subset=["pheno"])
tab5 = pd.crosstab(c5.subject_arm, c5.pheno)
print(tab5.to_string())
print("Fraction PR:", (c5.pheno == "PR").groupby(c5.subject_arm).mean().round(3).to_dict())
is_chr2 = c5.subject_arm == "ChR2"
ft = stats.fisher_exact([[((c5.pheno == "PR") & is_chr2).sum(), ((c5.pheno != "PR") & is_chr2).sum()],
                         [((c5.pheno == "PR") & ~is_chr2).sum(), ((c5.pheno != "PR") & ~is_chr2).sum()]])
print(f"Fisher exact, PR vs not, ChR2 vs both controls: odds ratio {ft.statistic:.2f}, p = {ft.pvalue:.3g}")
print("Fiber photometry RI60 mice, for reference:", (both.pheno.value_counts(normalize=True)).round(3).to_dict())

**Result: reproduced.** 12 of 13 ChR2 mice are PR (92%), against 5 of 7 EYFP (71%) and 6 of 12
ChR2 Scrambled (50%); Fisher's test of ChR2 against both controls gives p = 0.0497. The control
proportions are higher than in the fiber photometry cohort (19% PR).

### figs5f-chr2-pr-by-sex

> Under DMS ChR2 stimulation 100% of male mice and about 71% of female mice are PR.

In [ ]:
ch = c5[c5.subject_arm == "ChR2"]
print(pd.crosstab(ch.sex, ch.pheno).to_string())
print("Percent PR by sex:", (100 * (ch.pheno == "PR").groupby(ch.sex).mean()).round(1).to_dict(), "; n:", ch.sex.value_counts().to_dict())
print("ChR2 mice without both probes:", el5[(el5.subject_arm == "ChR2") & el5.pheno.isna()][["sex", "early", "late"]].to_dict("index"))

**Result: partially reproduced.** All 7 male ChR2 mice are PR (100%), as the paper says; 5 of 6
female ChR2 mice are PR (83.3%), outside the tolerance around 71%.

### fig5i-dms-stim-no-omission-effect

> DMS dopamine terminal stimulation does not change omission probe performance.

In [ ]:
def omission_anova(el):
    x = el.dropna(subset=["omission_min"])
    print(x.groupby("subject_arm").omission_min.agg(["mean", "std", "size", "min", "max"]).round(2).to_string())
    an = stats.f_oneway(*[g.omission_min for _, g in x.groupby("subject_arm")])
    kw = stats.kruskal(*[g.omission_min for _, g in x.groupby("subject_arm")])
    print(f"One-way ANOVA: F = {an.statistic:.2f}, p = {an.pvalue:.3g}; Kruskal-Wallis variant p = {kw.pvalue:.3g}")
    return an

an5 = omission_anova(el5)

**Result: reproduced.** Omission times do not differ across the DMS excitatory groups (ChR2 25.79,
Scrambled 35.53, EYFP 37.29 min; ANOVA p = 0.17).

### fig6-nphr-learning-dropout

> DMS dopamine terminal inhibition impairs initial learning; 25% of NpHR and 22% of NpHR Scrambled mice
> in group 1 failed FR1, and 29% of NpHR mice in group 2 failed RI30, and were dropped.

In [ ]:
s6, p6, el6 = opto["Opto-DMS-Inhibitory"]
print(el6.groupby("subject_arm").agg(mice=("n_ri60", "size"), did_not_learn=("did_not_learn", lambda x: (x == "True").sum()),
                                     no_ri60=("n_ri60", lambda x: (x == 0).sum()), both_probes=("pheno", "count")).to_string())
print("Mice flagged or without RI60:", el6[(el6.did_not_learn == "True") | (el6.n_ri60 == 0)][["subject_arm", "sex", "n_ri60", "did_not_learn"]].to_dict("index"))
stages6 = s6.groupby(["subject", "stage"]).size().unstack(fill_value=0)
print("Sessions per stage of those mice:", stages6.loc[el6[(el6.did_not_learn == "True") | (el6.n_ri60 == 0)].index].to_dict("index"))
first_fr1 = s6[s6.stage.isin(["FR1", "FR1habit"])].groupby("subject").start_ts.min()
first_ri30 = s6[s6.stage == "RI30"].groupby("subject").start_ts.min()
print("Mice whose first session is RI30 (light from RI30, possibly group 2):", int((first_ri30.reindex(el6.index) < first_fr1.reindex(el6.index)).sum()))
print("Session descriptions mentioning a group:", int(s6.description.str.contains("[Gg]roup").sum()))
print("FR1 sessions with optogenetic stimulation, by arm (all mice):", s6[s6.stage == "FR1"].groupby("subject_arm").size().to_dict())

**Result: not reproduced.** The dandiset flags two DMS inhibitory mice as not having learned: one
NpHR mouse (270.476, no RI60 session) and one EYFP mouse (244.465, 8 RI60 sessions and no probe). No
NpHR Scrambled mouse is flagged, where the paper drops 25% of group 1 NpHR, 22% of group 1 NpHR
Scrambled and 29% of group 2 NpHR mice. The dandiset holds 42 DMS inhibitory mice where Figure 6F counts
47, and no file says which group (1 or 2) a mouse belongs to; the dropped mice may be among the ones not
deposited.

### fig6g-nphr-fewer-shocks-late

> DMS dopamine terminal inhibition on rewarded nosepokes delays punishment resistance; NpHR mice accept
> fewer shocks than EYFP mice on the second probe (Tukey p < 0.01).

In [ ]:
long6 = p6[p6.probe != ""].merge(el6[["subject_arm"]].reset_index(), on="subject")
aov6, w6 = mixed_anova(long6, "subject_arm")
print(w6.groupby("subject_arm")[["early", "late"]].agg(["mean", "std", "size"]).round(2).to_string())
print("Second probe, Tukey:"); print(tukey(w6.late, w6.subject_arm).to_string())
print("Variant, two-way mixed ANOVA:"); print(fmt(aov6))
l6 = el6.dropna(subset=["late"])
print("Variant, all mice with a second probe:", l6.groupby("subject_arm").late.agg(["mean", "size"]).round(2).to_dict("index"))
print(tukey(l6.late, l6.subject_arm).to_string())

**Result: partially reproduced.** On the second probe NpHR mice accept 9.33 shocks and EYFP mice
23.47 (34 mice with both probes), but Tukey gives p = 0.069. Taking every mouse with a second probe (35)
gives p = 0.043; the two-way mixed ANOVA has group p = 0.056.

### fig6h-nphr-more-ps

> The NpHR group has more PS mice, and the NpHR Scrambled group more PR mice, than the EYFP group.

In [ ]:
c6 = el6.dropna(subset=["pheno"])
print(pd.crosstab(c6.subject_arm, c6.pheno).to_string())
fr = pd.crosstab(c6.subject_arm, c6.pheno, normalize="index").round(3)
print(fr.to_string())
def fisher(df, arm_a, arm_b, ph_):
    a, b = df[df.subject_arm == arm_a], df[df.subject_arm == arm_b]
    return stats.fisher_exact([[(a.pheno == ph_).sum(), (a.pheno != ph_).sum()], [(b.pheno == ph_).sum(), (b.pheno != ph_).sum()]]).pvalue
print(f"Fisher PS, NpHR vs EYFP: p = {fisher(c6, 'NpHR', 'EYFP', 'PS'):.3g}; PR, NpHRScrambled vs EYFP: p = {fisher(c6, 'NpHRScrambled', 'EYFP', 'PR'):.3g}")
print("By sex:"); print(pd.crosstab([c6.subject_arm, c6.sex], c6.pheno).to_string())

**Result: reproduced.** 8 of 9 NpHR mice are PS (89%) against 5 of 15 EYFP (33%; Fisher p = 0.013),
and 6 of 10 NpHR Scrambled mice are PR (60%) against 6 of 15 EYFP (40%; p = 0.43, not significant).

### fig6i-nphr-no-omission-effect

> DMS dopamine terminal inhibition does not change omission completion time.

In [ ]:
an6 = omission_anova(el6)

**Result: reproduced.** Omission times do not differ (EYFP 28.74, NpHR 21.47, NpHR Scrambled 27.82
min; ANOVA p = 0.21).

### fig7f-dls-escalation-similar

> Mice receiving DLS dopamine terminal stimulation (ChR2, EYFP, ChR2 Scrambled) escalate their nosepoking
> at the same rates.

In [ ]:
esc7, res7 = escalation("Opto-DLS-Excitatory")
for st, r in res7.items():
    print(st, f"{r['sessions']} sessions from {r['mice']} mice")
    print("  slope:", {k: round(v, 3) for k, v in r["slopes"].items()})
    print("  arm x index interaction against ChR2 (difference in slope, p):", {k: (round(v[0], 3), round(v[1], 4)) for k, v in r["interaction_vs_ref"].items()})
    print("  the third pair:", r["other_pair"][0], "difference in slope", round(r["other_pair"][1], 3), "p =", round(r["other_pair"][2], 4))
for st in ["FR1", "RI30"]:
    x = esc7[esc7.stage == st]
    f1 = smf.ols("rate ~ idx * C(arm)", x).fit(); f0 = smf.ols("rate ~ idx + C(arm)", x).fit()
    print(f"{st}: F test of all arm x index terms p = {f1.compare_f_test(f0)[1]:.3g}")

**Result: not reproduced.** In FR1 the ChR2 Scrambled slope (0.087 nosepokes/min per session) is
below the EYFP slope (0.224; interaction p = 0.013; all three groups F test p = 0.034). ChR2 does not
differ from either control (p = 0.12 and 0.98), and in RI30 no pair differs (p = 0.09 to 0.92).

### fig7g-dls-stim-no-shock-effect

> DLS dopamine terminal stimulation does not change the number of shocks received.

In [ ]:
_, p7, el7 = opto["Opto-DLS-Excitatory"]
long7 = p7[p7.probe != ""].merge(el7[["subject_arm"]].reset_index(), on="subject")
aov7, w7 = mixed_anova(long7, "subject_arm")
print(fmt(aov7))
print(w7.groupby("subject_arm")[["early", "late"]].agg(["mean", "std", "size"]).round(2).to_string())

**Result: reproduced.** Neither the group effect (p = 0.055) nor the group by probe interaction
(p = 0.068) is significant across ChR2, ChR2 Scrambled and EYFP mice (31 with both probes).

### fig7h-dls-similar-phenotypes

> DLS ChR2, EYFP and ChR2 Scrambled mice have similar distributions of PR, DPR and PS.

In [ ]:
c7 = el7.dropna(subset=["pheno"])
tab7 = pd.crosstab(c7.subject_arm, c7.pheno)
print(tab7.to_string())
chi = stats.chi2_contingency(tab7)
print(f"Chi-square: {chi.statistic:.2f}, df {chi.dof}, p = {chi.pvalue:.3g}; smallest expected count {chi.expected_freq.min():.1f}")

**Result: reproduced.** The phenotype distributions do not differ significantly (chi-square p = 0.12),
though the table is small (smallest expected count 1.5) and ChR2 mice include no DPR.

### fig7i-dls-no-omission-effect

> DLS dopamine terminal stimulation does not change omission completion time.

In [ ]:
an7 = omission_anova(el7)

**Result: reproduced.** Omission times do not differ (ChR2 28.44, Scrambled 33.51, EYFP 36.08 min;
ANOVA p = 0.53).

## Data outside the dandiset

### figs3ab-dat-protein-not-reduced

> Heterozygous DAT-IRES-cre mice do not have reduced DAT protein in DMS or DLS compared with wild-type
> mice.

The dandiset holds the western blot images (one per condition) and names the densitometry table on
Zenodo (doi:10.5281/zenodo.11266654) as its supplement. The table is read from Zenodo here.

In [ ]:
zen = requests.get("https://zenodo.org/api/records/11266654", timeout=60).json()
print("Zenodo record:", zen["doi"], "version", zen.get("metadata", {}).get("version"), "; files:", [f["key"] for f in zen["files"]])
xl = requests.get(zen["files"][0]["links"]["self"], timeout=60).content
print("Densitometry.xlsx md5:", hashlib.md5(xl).hexdigest(), "(Zenodo checksum", zen["files"][0]["checksum"] + ")")
sheet = pd.read_excel(io.BytesIO(xl), header=None)
blocks = {}
for i, label in sheet[0].items():
    if isinstance(label, str) and label.startswith(("Female", "Male")):
        rows = sheet.iloc[i + 1:i + 8]
        blocks[label.strip()] = pd.DataFrame({"sample": rows[0].str.strip().values, "corrected": rows[7].astype(float).values,
                                              "sheet_ratio": rows[8].values})
print("Blocks in the table:", list(blocks))
dat_rows = []
for region in ["DLS", "DMS"]:
    act, dat = blocks[f"Female {region} Actin"], blocks[f"Female {region} DAT"]
    ratio = dat.corrected.values / act.corrected.values
    geno = np.where(dat["sample"].str.startswith("WT"), "WT", "het")
    pct = 100 * ratio / ratio[geno == "WT"].mean()
    print(f"{region}: computed DAT/actin equals the sheet's ratio column: {np.allclose(ratio, dat.sheet_ratio.astype(float))}")
    for s_, g, p in zip(dat["sample"], geno, pct):
        dat_rows.append({"region": region, "sample": s_, "genotype": g, "pct_of_wt": p})
dd = pd.DataFrame(dat_rows)
print(dd.groupby(["region", "genotype"]).pct_of_wt.agg(["mean", "std", "size"]).round(1).to_string())
for region in ["DMS", "DLS"]:
    x = dd[dd.region == region]
    t3 = stats.ttest_ind(x[x.genotype == "het"].pct_of_wt, x[x.genotype == "WT"].pct_of_wt)
    mw = stats.mannwhitneyu(x[x.genotype == "het"].pct_of_wt, x[x.genotype == "WT"].pct_of_wt)
    print(f"{region}: het vs WT unpaired t = {t3.statistic:.2f}, p = {t3.pvalue:.3g}; Mann-Whitney p = {mw.pvalue:.3g}")
print("WT sample labels (one reads 'WT 4' where the others read 'WT F'):", sorted(dd[dd.genotype == "WT"]["sample"].unique()))
print("Male samples in the table:", int(dd["sample"].str.contains(" M ").sum()))

**Result: reproduced.** In the Zenodo table, DAT/actin of het mice is 131.3% of the WT mean in DMS
(t-test p = 0.41) and 74.7% in DLS (p = 0.20). The table holds females only (3 WT and 4 het samples per
region), and its sample names (410.1 to 410.3, 484.1 to 484.4) read as lanes of one WT and one het mouse;
the male samples are not in it.

## Claims that cannot be tested

- figs1a-shock-aversive (Figure S1A, fear conditioning in 13 wild-type mice): the dandiset holds no
  fear conditioning sessions or freezing scores; every subject belongs to the fiber photometry, the three
  optogenetic experiments, or the western blot.
- figs3cd-open-field (Figure S3C-D): no open field tracking is in the dandiset.
- fig3c-gcamp-in-dopamine-neurons (Figures 3C and 4C): no histology images or cell counts are in the
  dandiset.
- fig3e-ri30-dms-peaks: no RI30 photometry (see above).

In [ ]:
print("Session stages over all files:", sessions.stage.value_counts().to_dict())
print("Acquisition types in the western blot files: images only;",
      "files whose description mentions fear, freezing, open field or histology:",
      int(sessions.description.str.contains("fear|freez|open field|histolog", case=False).sum()))
print(f"Total runtime {time.time() - started:.0f} s")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| fig1d-ri60-vs-rr20-shocks | dandiset | not reproduced |
| fig1e-omission-ri60-longer | dandiset | partially reproduced |
| fig1f-omission-shock-no-correlation | dandiset | reproduced |
| fig2a-phenotype-rule | dandiset | partially reproduced |
| fig2b-only-dpr-increase | dandiset | reproduced |
| fig2c-nosepoke-rate | dandiset | reproduced |
| fig2d-omission-pr-vs-ps | dandiset | reproduced |
| fig2k-ps-wait-longer | dandiset | partially reproduced |
| fig2l-extra-nosepokes | dandiset | partially reproduced |
| fig2m-inter-reward-interval | dandiset | reproduced |
| fig2n-rewards-per-session | dandiset | reproduced |
| fig2o-escalation-timing | dandiset | partially reproduced |
| figs1i-ri60-more-rewards | dandiset | reproduced |
| figs2b-males-more-shocks | dandiset | reproduced |
| fig3e-ri30-dms-peaks | dandiset | blocked: data missing |
| fig3f-dms-score-phenotype | dandiset | not reproduced |
| fig3g-dms-score-shock-correlation | dandiset | partially reproduced |
| fig3h-dms-score-omission-no-correlation | dandiset | reproduced |
| fig4f-dls-score-no-phenotype | dandiset | reproduced |
| fig4g-dls-score-shock-no-correlation | dandiset | not reproduced |
| fig4h-dls-score-omission-no-correlation | dandiset | reproduced |
| fig5f-dms-chr2-escalation | dandiset | not reproduced |
| fig5g-dms-chr2-more-shocks-early | dandiset | not reproduced |
| fig5h-dms-chr2-mostly-pr | dandiset | reproduced |
| figs5f-chr2-pr-by-sex | dandiset | partially reproduced |
| fig5i-dms-stim-no-omission-effect | dandiset | reproduced |
| fig6-nphr-learning-dropout | dandiset | not reproduced |
| fig6g-nphr-fewer-shocks-late | dandiset | partially reproduced |
| fig6h-nphr-more-ps | dandiset | reproduced |
| fig6i-nphr-no-omission-effect | dandiset | reproduced |
| fig7f-dls-escalation-similar | dandiset | not reproduced |
| fig7g-dls-stim-no-shock-effect | dandiset | reproduced |
| fig7h-dls-similar-phenotypes | dandiset | reproduced |
| fig7i-dls-no-omission-effect | dandiset | reproduced |
| figs3ab-dat-protein-not-reduced | elsewhere (Zenodo) | reproduced |
| figs1a-shock-aversive | dandiset | blocked: data missing |
| figs3cd-open-field | dandiset | blocked: data missing |
| fig3c-gcamp-in-dopamine-neurons | dandiset | blocked: data missing |

How the analysis differs from the paper's. Statistics are run in Python, not Prism: mixed-effects
analyses are linear mixed models with a random intercept per mouse, and two-way repeated measures ANOVAs
use complete cases. Shock probe days with two files use the last file; mice with one probe day enter only
the analyses that need that probe. Photometry follows the authors' MATLAB code, with one added step that
leaves out channel dropouts (shown without it as a variant), and only early and late RI60 sessions are
available, so the RI30 stage of Figures 3F and 4F is missing. Optogenetic mice are classified with the
Methods rule, as the dandiset stores no phenotype for them. The definitions of the wait after reward,
extra nosepokes and nosepoke escalation are chosen here from the Results' wording. The supplementary
figure legends were not available.

What was found about the dandiset. The dandiset holds operant behavior for four experiments, fiber
photometry for RI60 and RR20 sessions, and western blot images. Missing against the paper: photometry
during RI30 and the shock probes; 3 of 9 RR20 mice; 6 of 39 DMS excitatory mice and 5 of 47
DMS inhibitory mice; one or both shock probes of several mice; the fear conditioning, open field and
histology data; any marker of the inhibitory experiment's groups 1 and 2. The 'Unknown' DLS excitatory
files repeat sessions deposited under the named arms, and one mouse (290.407) is filed under two
experiments. In left-side shock probes the reward series is named `right_reward_times`. One stitched
photometry session mixes MedPC and TDT clocks. Several photometry recordings hold dropouts that dominate
a z-score. Fiber coordinates are stored without the signs of the Methods. The fiber photometry mice carry
their phenotype in the file names and notes, which made Figure 2 testable; one stored label (048.392)
disagrees with the stated rule by one shock.